# Advancing BERT: NLP Programming Assignment 2 (Part 3)
**Objective**: Explore and improve key limitations of BERT-based models by assessing Tokenization gaps. Specifically, evaluating WordPiece (baseline) vs. BPE vs. Character-level tokenization for Question Answering on a SQuAD subset.

This notebook is structured to run end-to-end and will output metrics for your final discussion.

In [1]:
import sys
import os
# Prepend our custom D: drive PyTorch installation to sys.path
sys.path.insert(0, r"D:\-24-main\New folder\NLP_Assignment_2_BERT\pytorch_env")


In [ ]:
!pip install transformers datasets tokenizers evaluate accelerate pandas torch


In [2]:
import time
import os
import torch
import json
import pandas as pd
import evaluate
from datasets import load_dataset
from tokenizers import ByteLevelBPETokenizer
from tokenizers import Tokenizer, models, pre_tokenizers
from transformers import (
    BertTokenizerFast,
    PreTrainedTokenizerFast,
    BertForQuestionAnswering,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)


In [4]:
print("Loading SQuAD subset...")
dataset = load_dataset("squad")

# We use sub-sets for faster turnarounds: 2000 train, 500 val
# For char-level tokenization which is very expensive, we will use smaller subsets inside its block.
train_dataset = dataset["train"].select(range(2000))
val_dataset = dataset["validation"].select(range(500))

print(train_dataset[0])


Loading SQuAD subset...


{'id': '5733be284776f41900661182', 'title': 'University_of_Notre_Dame', 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.', 'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?', 'answers': {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}}


## Experiment 1: Baseline (WordPiece Tokenization)
Standard BERT uses WordPiece tokenization.


In [5]:
print("=== Starting Baseline Experiment (WordPiece) ===")
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

def preprocess(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=384,
        truncation="only_second",
        padding="max_length",
        return_offsets_mapping=True
    )

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(inputs["offset_mapping"]):
        answer = examples["answers"][i]
        if len(answer["answer_start"]) == 0:
            start_positions.append(0)
            end_positions.append(0)
            continue
            
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])

        sequence_ids = inputs.sequence_ids(i)

        idx = 0
        while idx < len(sequence_ids) and sequence_ids[idx] != 1:
            idx += 1
        context_start = idx

        while idx < len(sequence_ids) and sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        if context_start >= len(offsets) or offsets[context_start][0] > end_char or offsets[context_end][1] < start_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offsets[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx-1)

            idx = context_end
            while idx >= context_start and offsets[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx+1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    inputs.pop("offset_mapping")
    return inputs

train_baseline = train_dataset.map(preprocess, batched=True, remove_columns=train_dataset.column_names)
val_baseline = val_dataset.map(preprocess, batched=True, remove_columns=val_dataset.column_names)

model = BertForQuestionAnswering.from_pretrained("bert-base-uncased")

training_args = TrainingArguments(
    output_dir="./results_baseline",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="no"
)

data_collator = DataCollatorWithPadding(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_baseline,
    eval_dataset=val_baseline,
    processing_class=tokenizer,
    data_collator=data_collator
)

start_time = time.time()
trainer.train()
baseline_time = time.time() - start_time
print("Baseline Training Time:", baseline_time)

from transformers.utils.notebook import NotebookProgressCallback
if NotebookProgressCallback in [type(c) for c in trainer.callback_handler.callbacks]:
    trainer.remove_callback(NotebookProgressCallback)
eval_metrics_baseline = trainer.evaluate()
print("Baseline Evaluation Complete:", eval_metrics_baseline)

model.save_pretrained("bert_baseline")
size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, dn, filenames in os.walk("bert_baseline")
    for f in filenames
)
baseline_size = size / 1e6
print("Baseline Model Size (MB):", baseline_size)


=== Starting Baseline Experiment (WordPiece) ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
qa_outputs.bias                            | MISSING    | 
qa_outputs.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized beca

Step,Training Loss
50,4.898016
100,3.969239
150,3.404768
200,2.946974
250,2.795824


Baseline Training Time: 280.7421669960022
Baseline Evaluation Complete: {'eval_loss': 2.499000072479248, 'eval_runtime': 23.4782, 'eval_samples_per_second': 21.296, 'eval_steps_per_second': 2.683, 'epoch': 1.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Baseline Model Size (MB): 435.596855


## Experiment 2: BPE Tokenization
We learn common subword units by merging frequent character pairs.


In [6]:
print("=== Starting BPE Tokenizer Experiment ===")

texts = []
for item in dataset["train"].select(range(5000)):
    texts.append(item["context"])
    texts.append(item["question"])

os.makedirs("bpe_tokenizer", exist_ok=True)
with open("text.txt", "w", encoding="utf-8") as f:
    for t in texts:
        f.write(t + "\n")

bpe_tokenizer_train = ByteLevelBPETokenizer()
bpe_tokenizer_train.train(
    files=["text.txt"],
    vocab_size=30000,
    min_frequency=2
)
bpe_tokenizer_train.save("bpe_tokenizer/tokenizer.json")

bpe_tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="bpe_tokenizer/tokenizer.json",
    unk_token="[UNK]",
    pad_token="[PAD]",
    cls_token="[CLS]",
    sep_token="[SEP]",
    mask_token="[MASK]"
)

def preprocess_bpe(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = bpe_tokenizer(
        questions,
        examples["context"],
        max_length=384,
        truncation="only_second",
        padding="max_length",
        return_offsets_mapping=True
    )

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(inputs["offset_mapping"]):
        answer = examples["answers"][i]
        if len(answer["answer_start"]) == 0:
            start_positions.append(0)
            end_positions.append(0)
            continue
            
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])

        sequence_ids = inputs.sequence_ids(i)

        idx = 0
        while idx < len(sequence_ids) and sequence_ids[idx] != 1:
            idx += 1
        context_start = idx

        while idx < len(sequence_ids) and sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        if context_start >= len(offsets) or context_end < 0 or offsets[context_start][0] > end_char or offsets[context_end][1] < start_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offsets[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx-1)

            idx = context_end
            while idx >= context_start and offsets[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx+1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    if "offset_mapping" in inputs:
        inputs.pop("offset_mapping")
    return inputs

train_bpe = train_dataset.map(preprocess_bpe, batched=True, remove_columns=train_dataset.column_names)
val_bpe = val_dataset.map(preprocess_bpe, batched=True, remove_columns=val_dataset.column_names)

model_bpe = BertForQuestionAnswering.from_pretrained("bert-base-uncased")
model_bpe.resize_token_embeddings(len(bpe_tokenizer))

training_args_bpe = TrainingArguments(
    output_dir="./results_bpe",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="no"
)

data_collator_bpe = DataCollatorWithPadding(bpe_tokenizer)

trainer_bpe = Trainer(
    model=model_bpe,
    args=training_args_bpe,
    train_dataset=train_bpe,
    eval_dataset=val_bpe,
    processing_class=bpe_tokenizer,
    data_collator=data_collator_bpe
)

start_time = time.time()
trainer_bpe.train()
bpe_time = time.time() - start_time
print("BPE Training Time:", bpe_time)

if NotebookProgressCallback in [type(c) for c in trainer_bpe.callback_handler.callbacks]:
    trainer_bpe.remove_callback(NotebookProgressCallback)
eval_metrics_bpe = trainer_bpe.evaluate()

model_bpe.save_pretrained("bert_bpe")
size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, dn, filenames in os.walk("bert_bpe")
    for f in filenames
)
bpe_size = size / 1e6
print("BPE Model Size (MB):", bpe_size)


=== Starting BPE Tokenizer Experiment ===


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
qa_outputs.bias                            | MISSING    | 
qa_outputs.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized beca

Step,Training Loss
50,5.357435
100,4.927492
150,4.812293
200,4.726449
250,4.643887


BPE Training Time: 1629.8912997245789


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

BPE Model Size (MB): 419.057187


## Experiment 3: Character-level Tokenization
Instead of breaking into words or subwords, we break into distinct characters. We will evaluate Char-level tokenizer on a smaller SQuAD subset because the sequence length becomes extremely high (approx 4-5x longer vs subword approaches).


In [7]:
print("=== Starting Character-Level Tokenizer Experiment ===")

train_dataset_char = dataset["train"].select(range(500))
val_dataset_char = dataset["validation"].select(range(100))

chars = list("abcdefghijklmnopqrstuvwxyz0123456789.,!?;:-'\"() ")
vocab = {c: i+5 for i, c in enumerate(chars)}
vocab["[PAD]"] = 0
vocab["[UNK]"] = 1
vocab["[CLS]"] = 2
vocab["[SEP]"] = 3
vocab["[MASK]"] = 4

os.makedirs("char_tokenizer", exist_ok=True)
with open("char_tokenizer/vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f)

base_tokenizer = Tokenizer(models.BPE(vocab=vocab, merges=[]))
base_tokenizer.pre_tokenizer = pre_tokenizers.Split(pattern="", behavior="isolated")
base_tokenizer.save("char_tokenizer/tokenizer.json")

char_tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="char_tokenizer/tokenizer.json",
    unk_token="[UNK]",
    pad_token="[PAD]",
    cls_token="[CLS]",
    sep_token="[SEP]",
    mask_token="[MASK]"
)

def preprocess_char(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = char_tokenizer(
        questions,
        examples["context"],
        max_length=512, 
        truncation="only_second",
        padding="max_length",
        return_offsets_mapping=True
    )

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(inputs["offset_mapping"]):
        answer = examples["answers"][i]
        if len(answer["answer_start"]) == 0:
            start_positions.append(0)
            end_positions.append(0)
            continue
            
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])

        sequence_ids = inputs.sequence_ids(i)
        idx = 0
        while idx < len(sequence_ids) and sequence_ids[idx] != 1:
            idx += 1
        context_start = idx

        while idx < len(sequence_ids) and sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        if context_start >= len(offsets) or context_end < 0 or offsets[context_start][0] > end_char or offsets[context_end][1] < start_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offsets[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx-1)

            idx = context_end
            while idx >= context_start and offsets[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx+1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    if "offset_mapping" in inputs:
        inputs.pop("offset_mapping")
    return inputs

train_char_mapped = train_dataset_char.map(preprocess_char, batched=True, remove_columns=train_dataset_char.column_names)
val_char_mapped = val_dataset_char.map(preprocess_char, batched=True, remove_columns=val_dataset_char.column_names)

model_char = BertForQuestionAnswering.from_pretrained("bert-base-uncased")
model_char.resize_token_embeddings(len(char_tokenizer))

training_args_char = TrainingArguments(
    output_dir="./results_char",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="no"
)

data_collator_char = DataCollatorWithPadding(char_tokenizer)

trainer_char = Trainer(
    model=model_char,
    args=training_args_char,
    train_dataset=train_char_mapped,
    eval_dataset=val_char_mapped,
    processing_class=char_tokenizer,
    data_collator=data_collator_char
)

start_time = time.time()
trainer_char.train()
char_time = time.time() - start_time
print("Char Training Time:", char_time)

if NotebookProgressCallback in [type(c) for c in trainer_char.callback_handler.callbacks]:
    trainer_char.remove_callback(NotebookProgressCallback)
eval_metrics_char = trainer_char.evaluate()

model_char.save_pretrained("bert_char")
size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, dn, filenames in os.walk("bert_char")
    for f in filenames
)
char_size = size / 1e6
print("Char Model Size (MB):", char_size)


=== Starting Character-Level Tokenizer Experiment ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
qa_outputs.bias                            | MISSING    | 
qa_outputs.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized beca

Step,Training Loss
50,5.703688


Char Training Time: 1102.7508237361908


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Char Model Size (MB): 341.995948


## Evaluate all Results
We compile the extracted metrics (eval loss, training time, model size) over our subset tests.


In [8]:
results = pd.DataFrame({
    "Tokenizer": ["WordPiece", "BPE", "Character"],
    "Eval Loss": [eval_metrics_baseline.get("eval_loss", 0.0), eval_metrics_bpe.get("eval_loss", 0.0), eval_metrics_char.get("eval_loss", 0.0)],
    "Training Time (s)": [baseline_time, bpe_time, char_time],
    "Model Size (MB)": [baseline_size, bpe_size, char_size]
})

print("\n\n" + "="*50)
print("FINAL EXPERIMENT RESULTS")
print("="*50)
print(results.to_string(index=False))


results.to_csv("tokenization_results.csv", index=False)




FINAL EXPERIMENT RESULTS
Tokenizer  Eval Loss  Training Time (s)  Model Size (MB)
WordPiece   2.499000         280.742167       435.596855
      BPE   4.923283        1629.891300       419.057187
Character   6.340236        1102.750824       341.995948
